### In this notebook we show how the model to model evaluation module works

In [ ]:
import json

from model_evaluation import BPMNConverter, XMLBPMNConverter

In [ ]:
def load_model(path):
    """Load a BPMN model from a .json (Signavio) or .bpmn/.xml (BPMN 2.0) file.
    Returns the normalised dict ready for the evaluation pipeline.
    """
    if path.endswith(".xml") or path.endswith(".bpmn"):
        return XMLBPMNConverter.convert_file(path).to_dict()
    else:
        with open(path, "r", encoding="utf-8") as fh:
            raw = json.load(fh)
        return BPMNConverter.convert(raw).to_dict()


# Load the first model (P2P running example)
path_model1 = "../examples/p2p_running_example.bpmn"

# Load the second model (P2P variant running example)
path_model2 = "../examples/p2p_running_example_variant.bpmn"


model_1_json = load_model(path_model1)
model_2_json = load_model(path_model2)

## 3. Element statistics

Quick element-count summary for each model — activities, events, gateways, flows, pools, and lanes.

In [3]:
def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


for label, model in [("Model 1", model_1_json), ("Model 2", model_2_json)]:
    counts = count_elements(model)
    print(f"{label}: {sum(counts.values())} total elements")
    for key, val in counts.items():
        if val > 0:
            print(f"  • {key.replace('_', ' ').title()}: {val}")
    print()

Model 1: 48 total elements
  • Activities: 14
  • Events: 7
  • Gateways: 7
  • Sequence Flows: 16
  • Pools: 1
  • Lanes: 3

Model 2: 39 total elements
  • Activities: 13
  • Events: 5
  • Gateways: 5
  • Sequence Flows: 13
  • Pools: 1
  • Lanes: 2



## 4. Semantic name normalization

Align element names between the two models using a sentence-transformer-based string similarity (cosine), so semantically equivalent labels get unified before comparison. The threshold controls how aggressive the alignment is.

In [4]:
from model_evaluation import normalize_atomic_names
from model_evaluation.utils import cosine_sim_optimized

threshold = 0.6
print(f"Aligning element names (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(
    model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold
)

total_mappings = sum(len(v) for v in mappings.values())
if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            first_old, first_new = next(iter(mapping.items()))
            print(f"  • {elem_type}: {len(mapping)} mappings "
                  f"(e.g. '{first_old}' → '{first_new}')")
else:
    print("✓ No mappings needed (names already aligned)")


Aligning element names (threshold=0.6)...
✓ Applied 18 semantic name mappings
  • activity_names: 7 mappings (e.g. 'Release  payment' → 'Release  payment')
  • event_names: 2 mappings (e.g. 'Need Identified' → 'Need Identified')
  • pool_names: 1 mappings (e.g. 'SAMPLE Tech ORG' → 'SAMPLE ORG')
  • lane_names: 2 mappings (e.g. 'Finance & Procurement' → 'Purchasing')
  • activity_names__subprocess: 3 mappings (e.g. 'Auto-approval' → 'Auto-approval')
  • event_names__subprocess: 2 mappings (e.g. 'Request  denied' → 'Request denied')
  • gateway_names__subprocess: 1 mappings (e.g. 'amount' → 'amount > 5000€')


## 5. Trace extraction

Each model is converted to a Petri net and explored to enumerate the execution traces. Returns sound variants plus partial traces (deadlocks / timeouts), bounded by `timeout_seconds` and `max_loop_depth`.

In [5]:
from model_evaluation import extract_traces

tr1 = extract_traces(model_1_json, timeout_seconds=5, max_loop_depth=3)
tr2 = extract_traces(model2_aligned, timeout_seconds=5, max_loop_depth=3)

print(f"Model 1: {len(tr1.variants)} sound + {len(tr1.partial_traces)} partial traces")
print(f"Model 2: {len(tr2.variants)} sound + {len(tr2.partial_traces)} partial traces")

print("\nFirst 3 traces from Model 1:")
for t in tr1.all_traces()[:3]:
    print(" ", t)
print("\nFirst 3 traces from Model 2:")
for t in tr2.all_traces()[:3]:
    print(" ", t)


Trace extraction recovered partial results for net '<unnamed>': 34977 sound variant(s), 78866 partial trace(s); 158815 loop-cap hit(s), exploration timed out, exploration truncated by active-set cap
Trace extraction recovered partial results for net '<unnamed>': 23833 sound variant(s), 94258 partial trace(s); 246843 loop-cap hit(s), exploration timed out, exploration truncated by active-set cap


Model 1: 34977 sound + 78866 partial traces
Model 2: 23833 sound + 94258 partial traces

First 3 traces from Model 1:
  ['Auto-approval', 'Management review', 'Perform 3-way match', 'Auto-approval', 'Auto-approval', 'Management review', 'Check for approval type', 'Check invoice data for completeness', 'Check for approval type', 'StartNoneEvent']
  ['Perform 3-way match', 'Management review', 'Perform 3-way match', 'Perform 3-way match', 'Auto-approval', 'Check for approval type', 'StartNoneEvent', 'Auto-approval', 'Auto-approval', 'Check for approval type', 'Check invoice data for completeness']
  ['Auto-approval', 'Auto-approval', 'Check invoice data for completeness', 'Perform 3-way match', 'Auto-approval', 'Check for approval type', 'Check for approval type', 'StartNoneEvent', 'Check for approval type', 'Perform 3-way match']

First 3 traces from Model 2:
  ['Management review', 'Team Lead Review', 'Management review', 'Team Lead Review', 'Team Lead Review', 'Auto-approval', 'Auto-a

## 6. Structural / behavioral / hybrid similarity

- **Structural**: weighted score across element / flow / organizational / subprocess categories.
- **Behavioral**: how much the trace sets overlap (set-based metric over whole traces).
- **Hybrid**: a weighted combination of the two.

In [6]:
from model_evaluation import (
    calculate_bpmn_similarity,
    calculate_trace_similarity,
    calculate_hybrid_similarity,
)

struct = calculate_bpmn_similarity(model_1_json, model2_aligned, method="jaccard")
beh = calculate_trace_similarity(tr1, tr2, method="jaccard")
hybrid = calculate_hybrid_similarity(struct, beh, structural_weight=0.5)
print(hybrid)


{'structural': 0.49973134708428824, 'behavioral': 0.0028308210488995253, 'structural_weight': 0.5, 'behavioral_weight': 0.5, 'hybrid': 0.2512810840665939}


## 7. N-gram trace comparison

N-grams are length-`n` contiguous activity sequences extracted from each trace. Each trace is wrapped in `<START>`/`<END>` tokens so trace boundaries become distinct n-grams.

- `n=1` → unigrams (multiset of activities)
- `n=2` → bigrams / directly-follows pairs
- larger `n` → longer behavioral motifs

Set-based similarity (jaccard / dice / overlap) on the n-gram sets gives a graded view of behavioral agreement that isn’t as harsh as exact-variant matching.

In [7]:
from collections import Counter
from model_evaluation import calculate_ngram_similarity, extract_ngrams

for n in (1, 2, 3):
    print(f"\nn={n}")
    for method in ("jaccard", "dice", "overlap"):
        score = calculate_ngram_similarity(tr1, tr2, n=n, method=method)
        print(f"  {method:8s}: {score:.2%}")
    c1 = Counter(extract_ngrams(tr1, n=n))
    c2 = Counter(extract_ngrams(tr2, n=n))
    shared = c1.keys() & c2.keys()
    only_1 = c1.keys() - c2.keys()
    only_2 = c2.keys() - c1.keys()
    print(f"  shared: {len(shared)}  |  only in M1: {len(only_1)}  |  only in M2: {len(only_2)}")



n=1
  jaccard : 52.00%
  dice    : 68.42%
  overlap : 72.22%
  shared: 13  |  only in M1: 5  |  only in M2: 7

n=2
  jaccard : 30.86%
  dice    : 47.17%
  overlap : 48.08%
  shared: 75  |  only in M1: 87  |  only in M2: 81

n=3
  jaccard : 19.27%
  dice    : 32.32%
  overlap : 36.16%
  shared: 328  |  only in M1: 795  |  only in M2: 579


## 8. Interactive dashboard

The interactive dashboard lives in [`notebooks/dashboard.py`](dashboard.py) as a marimo notebook. It bundles the structural, behavioral (with the n-gram subpanel), and hybrid sections into one reactive view.

Launch it from the repo root with:

```
poetry run marimo edit notebooks/dashboard.py
```
